# Human vs AI Essay Analysis: Memory Curve Comparison

**Question:** Do AI-generated essays show different long-range coherence patterns than human-written essays?

**Design:**
- Human essays: sampled from PERSUADE 2.0 (real student argumentative essays)
- AI essays: generated by Claude on the same 5 prompts
- Both analyzed with context ablation using Mistral-7B
- Compare memory curves, half-lives, AUC, and curve shapes

In [ ]:
# ── Install & Import ──
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from pathlib import Path
import json, math, time, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
# ── Config ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/HumanVsAI")
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/human_vs_ai")
    IN_COLAB = True
except:
    BASE_DIR = Path("../results/human_vs_ai")
    DATA_DIR = Path("../data/human_vs_ai")
    IN_COLAB = False

BASE_DIR.mkdir(parents=True, exist_ok=True)

# Model config
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4/V100, False for A100

# Dense window config — extra granularity at short contexts where most benefit occurs
WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256]
BURN_IN = 256
MAX_SCORE_TOKENS = 128  # Shorter scoring region to keep total sequence manageable
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER  # 394

print(f"Data dir: {DATA_DIR}")
print(f"Results dir: {BASE_DIR}")
print(f"Windows: {WINDOWS}")
print(f"Min tokens required: {MIN_TOKENS}")

In [ ]:
# ── Load corpus ──
corpus_path = DATA_DIR / "human_vs_ai_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))

human_docs = [d for d in corpus if d["population"] == "human"]
ai_docs = [d for d in corpus if d["population"] == "ai"]

print(f"Loaded {len(corpus)} essays: {len(human_docs)} human, {len(ai_docs)} AI")
print(f"\nPer prompt:")
for slug in sorted(set(d["prompt_slug"] for d in corpus)):
    nh = sum(1 for d in human_docs if d["prompt_slug"] == slug)
    na = sum(1 for d in ai_docs if d["prompt_slug"] == slug)
    print(f"  {slug:<25} human={nh}, ai={na}")

In [ ]:
# ── Load model ──
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

## Compute Memory Curves

In [ ]:
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens[target_start:target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return float('inf'), 0

    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]

    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        token_loss = -log_probs[token_ids[i + 1]].item()
        total_loss += token_loss
        count += 1

    return math.exp(total_loss / count) if count > 0 else float('inf'), count


def compute_memory_curve(token_ids, windows, burn_in, max_score_tokens):
    """Compute perplexity at each context window size."""
    result = {'ppl_by_W': {}, 'token_count': len(token_ids)}
    n_tokens = len(token_ids)

    if n_tokens <= burn_in:
        return result

    target_end = min(n_tokens, burn_in + max_score_tokens)

    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        if actual_context < 4:
            continue

        truncated = token_ids[context_start:target_end]
        ppl, _ = compute_perplexity_on_region(truncated, actual_context, len(truncated))

        if not math.isinf(ppl):
            result['ppl_by_W'][W] = ppl

    return result


def compute_half_life(ppl_dict, percentile=0.5):
    """Context length where percentile of total benefit is achieved."""
    if len(ppl_dict) < 2:
        return float('nan')
    items = sorted(ppl_dict.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return float('nan')
    target_ppl = ppls[0] - percentile * total_benefit
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    return windows[-1]


print("Functions defined")

In [ ]:
# ── Run memory curves on all essays ──
results = []
skipped = 0

for doc in tqdm(corpus, desc="Computing memory curves"):
    token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
    n_tokens = len(token_ids)

    if n_tokens < MIN_TOKENS:
        skipped += 1
        continue

    curve = compute_memory_curve(token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)

    if len(curve['ppl_by_W']) < 3:
        skipped += 1
        continue

    ppl_by_W = curve['ppl_by_W']

    # Build result row with ppl at each window
    row = {
        'doc_id': doc['doc_id'],
        'population': doc['population'],
        'prompt_slug': doc['prompt_slug'],
        'token_count': n_tokens,
        'half_life': compute_half_life(ppl_by_W),
    }

    for W in WINDOWS:
        row[f'ppl_W{W}'] = ppl_by_W.get(W, np.nan)

    # Derived metrics using full curve
    computed_windows = sorted([w for w in WINDOWS if w in ppl_by_W])
    computed_ppls = np.array([ppl_by_W[w] for w in computed_windows])
    delta_ppl = computed_ppls[0] - computed_ppls  # improvement from shortest context

    row['delta_max'] = delta_ppl[-1]
    row['auc_full'] = trapezoid(delta_ppl, computed_windows)

    # Log-slope across full range
    if len(computed_windows) >= 3:
        slope, _, _, _, _ = stats.linregress(np.log(computed_windows), delta_ppl)
        row['log_slope'] = slope

    # Early vs late: split at 32 tokens
    early_W = [w for w in computed_windows if w <= 32]
    late_W = [w for w in computed_windows if w >= 32]
    if len(early_W) >= 2:
        early_ppls = np.array([ppl_by_W[w] for w in early_W])
        row['auc_early'] = trapezoid(early_ppls[0] - early_ppls, early_W)
    if len(late_W) >= 2:
        late_ppls = np.array([ppl_by_W[w] for w in late_W])
        row['auc_late'] = trapezoid(late_ppls[0] - late_ppls, late_W)

    # Early fraction: what % of total AUC comes from first 32 tokens of context
    if row.get('auc_early') and row.get('auc_full') and row['auc_full'] > 0:
        row['early_fraction'] = row['auc_early'] / row['auc_full']

    results.append(row)

df = pd.DataFrame(results)
print(f"\nProcessed {len(df)} essays ({skipped} skipped)")
print(f"  Human: {len(df[df.population == 'human'])}")
print(f"  AI:    {len(df[df.population == 'ai'])}")

# Save raw results
df.to_csv(BASE_DIR / 'essay_level_results.csv', index=False)
print(f"\nSaved to {BASE_DIR / 'essay_level_results.csv'}")

## Descriptive Statistics & Length Check

In [ ]:
# ── Descriptive stats ──
COLORS = {'human': '#3498db', 'ai': '#e74c3c'}

print("="*70)
print("DESCRIPTIVE STATISTICS BY POPULATION")
print("="*70)

key_cols = ['token_count', 'ppl_W4', 'ppl_W32', 'ppl_W128', 'ppl_W256',
            'half_life', 'delta_max', 'auc_full', 'auc_early', 'auc_late',
            'early_fraction', 'log_slope']
key_cols = [c for c in key_cols if c in df.columns]

desc = df.groupby('population')[key_cols].agg(['mean', 'std', 'median'])
print(desc.round(3).to_string())

# Length balance check
print("\n\nLENGTH BALANCE:")
for pop in ['human', 'ai']:
    sub = df[df.population == pop]
    toks = sub['token_count']
    print(f"  {pop}: median={toks.median():.0f}, mean={toks.mean():.0f}, "
          f"range=[{toks.min()}, {toks.max()}]")

t_len, p_len = stats.ttest_ind(
    df[df.population == 'human']['token_count'],
    df[df.population == 'ai']['token_count']
)
print(f"  Length difference: t={t_len:.2f}, p={p_len:.4f}")
if p_len < 0.05:
    print("  WARNING: Significant length difference — will control for this")

## Memory Curve Comparison

In [ ]:
# ── Figure 1: Dense memory curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Collect ppl columns that exist
ppl_cols = [f'ppl_W{w}' for w in WINDOWS if f'ppl_W{w}' in df.columns]
plot_windows = [int(c.replace('ppl_W', '')) for c in ppl_cols]

# 1a: Overall mean curves with SEM bands
ax = axes[0]
for pop in ['human', 'ai']:
    sub = df[df.population == pop]
    means = [sub[c].mean() for c in ppl_cols]
    sems = [sub[c].sem() for c in ppl_cols]
    means, sems = np.array(means), np.array(sems)
    ax.plot(plot_windows, means, marker='o', label=pop,
            color=COLORS[pop], linewidth=2, markersize=5)
    ax.fill_between(plot_windows, means - sems, means + sems,
                    color=COLORS[pop], alpha=0.15)

ax.set_xlabel('Context Window (tokens)', fontsize=12)
ax.set_ylabel('Perplexity', fontsize=12)
ax.set_title('Mean Memory Curves: Human vs AI', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# 1b: Normalized improvement curves (delta from shortest context)
ax = axes[1]
for pop in ['human', 'ai']:
    sub = df[df.population == pop]
    means = np.array([sub[c].mean() for c in ppl_cols])
    # Normalize: 0 = no improvement, 1 = full improvement
    total_drop = means[0] - means[-1]
    if total_drop > 0:
        normalized = (means[0] - means) / total_drop
    else:
        normalized = np.zeros_like(means)
    ax.plot(plot_windows, normalized, marker='o', label=pop,
            color=COLORS[pop], linewidth=2, markersize=5)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% benefit')
ax.set_xlabel('Context Window (tokens)', fontsize=12)
ax.set_ylabel('Fraction of Total Benefit', fontsize=12)
ax.set_title('Normalized Improvement Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_memory_curves_dense.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Metric distributions ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics_to_plot = [
    ('ppl_W4', 'Baseline Perplexity (W4)'),
    ('half_life', 'Half-Life (tokens)'),
    ('auc_full', 'AUC (full curve)'),
    ('auc_early', 'AUC Early (4→32)'),
    ('early_fraction', 'Early Fraction (AUC early/full)'),
    ('log_slope', 'Log Slope'),
]

for idx, (col, label) in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    if col not in df.columns:
        ax.set_title(f'{label}\n(not computed)', fontsize=11)
        continue

    for pop in ['human', 'ai']:
        sub = df[df.population == pop][col].dropna()
        ax.hist(sub, bins=25, alpha=0.5, color=COLORS[pop], label=pop, density=True)

    h = df[df.population == 'human'][col].dropna()
    a = df[df.population == 'ai'][col].dropna()
    t, p = stats.ttest_ind(h, a)
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)

    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.set_title(f'{label}\nd={d:.2f}, p={p:.4f} {sig}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Metric Distributions: Human vs AI', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests

In [ ]:
# ── Omnibus tests ──
print("="*70)
print("UNIVARIATE TESTS: Human vs AI")
print("="*70)

test_metrics = ['ppl_W4', 'ppl_W32', 'ppl_W128', 'ppl_W256',
                'half_life', 'delta_max', 'auc_full', 'auc_early', 'auc_late',
                'early_fraction', 'log_slope']
test_metrics = [m for m in test_metrics if m in df.columns]

print(f"\n{'Metric':<20} {'Human mean':>12} {'AI mean':>12} {'Cohen d':>10} {'t':>8} {'p':>10}")
print("-"*75)

for m in test_metrics:
    h = df[df.population == 'human'][m].dropna()
    a = df[df.population == 'ai'][m].dropna()
    t, p = stats.ttest_ind(h, a)
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{m:<20} {h.mean():>12.3f} {a.mean():>12.3f} {d:>10.3f} {t:>8.2f} {p:>9.4f} {sig}")

In [ ]:
# ── Regression: control for length and baseline fluency ──
print("="*70)
print("REGRESSION: Does population predict curve shape after controls?")
print("="*70)

def standardize(s):
    return (s - s.mean()) / s.std()

df['token_count_z'] = standardize(df['token_count'])
df['ppl_W4_z'] = standardize(df['ppl_W4'])
df['is_ai'] = (df['population'] == 'ai').astype(int)

KEY_METRICS = ['auc_full', 'auc_early', 'early_fraction', 'half_life', 'log_slope']
KEY_METRICS = [m for m in KEY_METRICS if m in df.columns]

for metric in KEY_METRICS:
    print(f"\n{'='*60}")
    print(f"{metric.upper()}")
    print(f"{'='*60}")

    # Model A: population + length
    formula_a = f'{metric} ~ is_ai + token_count_z'
    model_a = smf.ols(formula_a, data=df.dropna(subset=[metric])).fit()

    # Model B: + baseline fluency (ppl at shortest context)
    formula_b = f'{metric} ~ is_ai + token_count_z + ppl_W4_z'
    model_b = smf.ols(formula_b, data=df.dropna(subset=[metric])).fit()

    # Model C: + prompt (fixed effect)
    formula_c = f'{metric} ~ is_ai + token_count_z + ppl_W4_z + C(prompt_slug)'
    model_c = smf.ols(formula_c, data=df.dropna(subset=[metric])).fit()

    print(f"\n{'Model':<10} {'R2':>8} {'B(AI)':>10} {'p(AI)':>10} {'Sig':>5}")
    print("-"*48)
    for name, m in [('A: +len', model_a), ('B: +ppl', model_b), ('C: +prompt', model_c)]:
        beta = m.params.get('is_ai', np.nan)
        p = m.pvalues.get('is_ai', np.nan)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{name:<10} {m.rsquared:>8.4f} {beta:>10.4f} {p:>10.4f} {sig:>5}")

## Per-Prompt Breakdown

In [ ]:
# ── Figure 3: Per-prompt metric comparison ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
prompts = sorted(df['prompt_slug'].unique())

plot_metrics = [
    ('auc_full', 'AUC (full)'), ('early_fraction', 'Early Fraction'),
    ('half_life', 'Half-Life'), ('ppl_W4', 'Baseline Perplexity (W4)')
]

for idx, (metric, label) in enumerate(plot_metrics):
    ax = axes[idx // 2, idx % 2]
    if metric not in df.columns:
        continue
    x = np.arange(len(prompts))
    width = 0.35

    h_means = [df[(df.population == 'human') & (df.prompt_slug == p)][metric].mean() for p in prompts]
    a_means = [df[(df.population == 'ai') & (df.prompt_slug == p)][metric].mean() for p in prompts]
    h_sems = [df[(df.population == 'human') & (df.prompt_slug == p)][metric].sem() for p in prompts]
    a_sems = [df[(df.population == 'ai') & (df.prompt_slug == p)][metric].sem() for p in prompts]

    ax.bar(x - width/2, h_means, width, yerr=h_sems, label='Human',
           color=COLORS['human'], alpha=0.8, capsize=3)
    ax.bar(x + width/2, a_means, width, yerr=a_sems, label='AI',
           color=COLORS['ai'], alpha=0.8, capsize=3)

    ax.set_ylabel(label, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace('_', '\n') for p in prompts], fontsize=8)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_title(label, fontsize=12, fontweight='bold')

plt.suptitle('Per-Prompt Comparison: Human vs AI', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_per_prompt.png', dpi=150, bbox_inches='tight')
plt.show()

## Scatter: Curve Shape vs Baseline Fluency

In [ ]:
# ── Figure 4: Curve shape vs baseline fluency ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter_metrics = [('auc_full', 'AUC (full)'), ('early_fraction', 'Early Fraction')]

for idx, (metric, label) in enumerate(scatter_metrics):
    ax = axes[idx]
    if metric not in df.columns:
        continue
    for pop in ['human', 'ai']:
        sub = df[df.population == pop]
        ax.scatter(sub['ppl_W4'], sub[metric], c=COLORS[pop], label=pop,
                   alpha=0.6, s=50, edgecolors='white', linewidth=0.5)

    for pop, ls in [('human', '-'), ('ai', '--')]:
        sub = df[df.population == pop].dropna(subset=['ppl_W4', metric])
        if len(sub) < 3:
            continue
        z = np.polyfit(sub['ppl_W4'], sub[metric], 1)
        p = np.poly1d(z)
        x_line = np.linspace(sub['ppl_W4'].min(), sub['ppl_W4'].max(), 100)
        ax.plot(x_line, p(x_line), color=COLORS[pop], linestyle=ls, linewidth=2, alpha=0.8)

    ax.set_xlabel('Baseline Perplexity (ppl_W4)', fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'{label} vs Baseline Fluency', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Do human and AI essays follow different fluency-shape relationships?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig4_fluency_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
# ── Summary findings ──
print("="*70)
print("FINDINGS SUMMARY: Human vs AI Memory Curves")
print("="*70)

n_human = len(df[df.population == 'human'])
n_ai = len(df[df.population == 'ai'])

print(f"\nSample: {n_human} human essays, {n_ai} AI essays")
print(f"Prompts: {len(df.prompt_slug.unique())}")
print(f"Model: {MODEL_NAME}")
print(f"Windows: {WINDOWS} ({len(WINDOWS)} points)")

summary_metrics = ['ppl_W4', 'auc_full', 'auc_early', 'early_fraction', 'half_life', 'log_slope']
summary_metrics = [m for m in summary_metrics if m in df.columns]

print("\nKey comparisons:")
for m in summary_metrics:
    h = df[df.population == 'human'][m]
    a = df[df.population == 'ai'][m]
    t, p = stats.ttest_ind(h.dropna(), a.dropna())
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
    direction = "human > ai" if h.mean() > a.mean() else "ai > human"
    print(f"  {m:<20} {direction:<15} d={d:+.3f}  p={p:.4f}")

df.to_csv(BASE_DIR / 'essay_level_results.csv', index=False)
print(f"\nAll results saved to {BASE_DIR}/")